<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/05_Hands_On_Labs/notebooks/10_pytorch_intro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PyTorch Fundamentals: Tensors, Autograd & Training

> **Track:** ML Engineer | **Level:** Intermediate | **Time:** 2-3 hours

## Overview
PyTorch is the dominant framework for ML research and production. This notebook covers everything from tensor basics to building and training a neural net with a proper training loop.

### What You'll Learn
- Tensor creation and operations
- GPU vs CPU tensors
- Autograd: automatic differentiation
- nn.Module architecture
- DataLoader and custom Dataset
- Training loop (forward, loss, backward, step)
- Saving and loading models

```bash
pip install torch torchvision numpy matplotlib
```

In [ ]:
# Setup: tensors and device configuration
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch version: {torch.__version__}')
print(f'Using device: {device}')

# Tensor creation
a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
b = torch.zeros(3, 4)
c = torch.randn(2, 3)  # from standard normal
d = torch.arange(0, 10, 2, dtype=torch.float32)
print(f'\nTensor a:\n{a}')
print(f'Tensor operations: a.T =\n{a.T}')
print(f'Element-wise multiply: a * a =\n{a * a}')
print(f'Matrix multiply: a @ a.T =\n{a @ a.T}')

## 1. Autograd: Automatic Differentiation

PyTorch tracks operations on tensors with `requires_grad=True` and computes gradients via `.backward()`.

In [ ]:
# Autograd demonstration

# Simple example: y = x^2 + 3x + 1, dy/dx = 2x + 3
x = torch.tensor(2.0, requires_grad=True)
y = x**2 + 3*x + 1
y.backward()
print(f'y = x² + 3x + 1 at x=2: y={y.item():.1f}')
print(f'dy/dx at x=2: {x.grad.item():.1f} (expected: {2*2+3})')

# Chain rule in a small network
W1 = torch.randn(3, 4, requires_grad=True)
W2 = torch.randn(4, 2, requires_grad=True)
x_in = torch.randn(5, 3)  # batch of 5, input dim 3
target = torch.zeros(5, 2)

# Forward
h = torch.relu(x_in @ W1)  # (5,4)
out = h @ W2  # (5,2)
loss = ((out - target)**2).mean()

# Backward
loss.backward()
print(f'\nLoss: {loss.item():.4f}')
print(f'W1 gradient shape: {W1.grad.shape}')
print(f'W2 gradient shape: {W2.grad.shape}')

## 2. Building a Neural Network with nn.Module

In [ ]:
# nn.Module: the standard way to define models in PyTorch

class MLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int, dropout: float = 0.2):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, output_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)

model = MLP(input_dim=20, hidden_dim=64, output_dim=1).to(device)
print('Model architecture:')
print(model)
print(f'\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## 3. DataLoader and Custom Dataset

In [ ]:
# Custom Dataset and DataLoader for batch training

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler as SKScaler

class TabularDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y).unsqueeze(1)

    def __len__(self) -> int:
        return len(self.X)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.X[idx], self.y[idx]

# Generate classification dataset
X, y = make_classification(n_samples=1000, n_features=20, n_informative=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = SKScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

train_ds = TabularDataset(X_train, y_train)
test_ds = TabularDataset(X_test, y_test)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64)

print(f'Training samples: {len(train_ds)} | Batches per epoch: {len(train_loader)}')
print(f'Test samples: {len(test_ds)}')
Xb, yb = next(iter(train_loader))
print(f'Batch shapes: X={Xb.shape}, y={yb.shape}')

## 4. Training Loop and Evaluation

In [ ]:
# Complete training loop with validation

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for Xb, yb in loader:
        Xb, yb = Xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = model(Xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(Xb)
    return total_loss / len(loader.dataset)

def evaluate(model, loader, device):
    model.eval()
    correct = 0
    with torch.no_grad():
        for Xb, yb in loader:
            Xb, yb = Xb.to(device), yb.to(device)
            preds = (torch.sigmoid(model(Xb)) > 0.5).float()
            correct += (preds == yb).sum().item()
    return correct / len(loader.dataset)

model = MLP(20, 64, 1).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

train_losses = []
for epoch in range(30):
    loss = train_epoch(model, train_loader, optimizer, criterion, device)
    scheduler.step()
    train_losses.append(loss)
    if epoch % 10 == 0:
        acc = evaluate(model, test_loader, device)
        print(f'Epoch {epoch:3d} | Loss: {loss:.4f} | Test Acc: {acc:.4f}')

# Save and reload model
torch.save(model.state_dict(), 'model.pt')
model2 = MLP(20, 64, 1).to(device)
model2.load_state_dict(torch.load('model.pt', map_location=device))
print(f'\nFinal test accuracy: {evaluate(model2, test_loader, device):.4f}')
print('Model saved and reloaded successfully!')

## Exercises

1. **Learning rate scheduler**: Replace `StepLR` with `CosineAnnealingLR` (warmup for 5 epochs then cosine decay). Plot the learning rate schedule over 30 epochs and compare final test accuracy with StepLR.

2. **Early stopping**: Implement early stopping that halts training if validation loss doesn't improve for 5 consecutive epochs. Track the best model checkpoint and restore it at the end.

3. **Convolutional network**: Modify the architecture to use `nn.Conv2d` layers for image data. Load MNIST via `torchvision.datasets.MNIST`, build a small CNN (2 conv + 2 linear layers), and achieve at least 98% test accuracy.